<a href="https://colab.research.google.com/github/juuuuuuuuuuuuuun/pytorch-cifar10/blob/main/sutdy_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

データの準備


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
# Download training data from open datasets
#CIFAR10はカラー画像の学習セット
from google.colab import drive
drive.mount("/content/drive")

train_transform = v2.Compose([
        v2.RandomCrop(32, padding=4),
        v2.RandomHorizontalFlip(),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

test_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

training_data = datasets.CIFAR10(
    root="/content/drive/MyDrive/NN_study/CIFAR10",
    train=True,
    download=True,
    transform=train_transform,#AIが計算しやすい形に加工する
)

# Download test data from open datasets.
test_data = datasets.CIFAR10(
    root="/content/drive/MyDrive/NN_study/CIFAR10",
    train=False,
    download=True,
    transform=test_transform,
)


# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Mounted at /content/drive
Shape of X [N, C, H, W]: torch.Size([64, 3, 32, 32])
Shape of y: torch.Size([64]) torch.int64


ニューラルネットワークの準備

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

X, y = next(iter(train_dataloader))
out = model(X.to(device))
print(out.shape)  # torch.Size([64, 10]) と出れば成功


Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=3072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
torch.Size([64, 10])


学習してテストする

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    total_loss = 0  # ← 追加：損失を貯める変数
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)
        total_loss += loss.item()  # ← 追加：バッチごとに足す

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return total_loss / len(dataloader)  # ← 追加：1エポックの平均損失を返す

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return 100 * correct  # ← 追加：精度（%）を返す

train_losses = []  # ← 追加：記録用のリスト
test_accs = []     # ← 追加

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    loss = train(train_dataloader, model, loss_fn, optimizer)  # ← 戻り値を受け取る
    acc = test(test_dataloader, model, loss_fn)                 # ← 戻り値を受け取る
    train_losses.append(loss)  # ← 追加：リストに貯める
    test_accs.append(acc)      # ← 追加
print("Done!")
print(train_losses)
print(test_accs)


Epoch 1
-------------------------------
loss: 1.606310  [   64/50000]
loss: 1.535482  [ 6464/50000]
loss: 1.473451  [12864/50000]
loss: 1.645459  [19264/50000]
loss: 1.535950  [25664/50000]
loss: 1.440064  [32064/50000]
loss: 1.577931  [38464/50000]
loss: 1.399011  [44864/50000]
Test Error: 
 Accuracy: 42.0%, Avg loss: 1.614329 

Epoch 2
-------------------------------
loss: 1.485043  [   64/50000]
loss: 1.381131  [ 6464/50000]
loss: 1.463559  [12864/50000]
loss: 1.631197  [19264/50000]
loss: 1.659300  [25664/50000]
loss: 1.457742  [32064/50000]
loss: 1.605710  [38464/50000]
loss: 1.617016  [44864/50000]
Test Error: 
 Accuracy: 42.7%, Avg loss: 1.625464 

Epoch 3
-------------------------------
loss: 1.324155  [   64/50000]
loss: 1.342673  [ 6464/50000]
loss: 1.421888  [12864/50000]
loss: 1.427964  [19264/50000]
loss: 1.467731  [25664/50000]
loss: 1.623422  [32064/50000]
loss: 1.501183  [38464/50000]
loss: 1.542689  [44864/50000]
Test Error: 
 Accuracy: 44.2%, Avg loss: 1.555739 

Epoc

学習したモデルを生成している

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x1024 and 3072x512)